# 🏁 Podium Predictions — experimental sandbox

Your workspace for the **experimental** podium-prediction model. The project gives you the plumbing;
the modelling is yours.

**Provided for you** (in `f1pitwall.predictions`):
- `load_results(years, cache_dir)` — FastF1 race results as a tidy DataFrame.
- `latest_driver_lineup(results)` — entry list for the next race.
- `build_predictions(...)` / `export_predictions(...)` — turn your model's podium probabilities into
  the validated `public-data/predictions/next.json` that the dashboard's "Experimental Predictions"
  section renders.

**Yours to build**: feature engineering, model, evaluation, and the per-driver probabilities.

### Ground rules (so the results mean something)
- **No leakage.** Features for a race must use only *earlier* races — shift every rolling window by one.
- **Evaluate race-wise.** Split by race (`GroupKFold` on `chrono`), never a random row split.
- **Rank-aware metrics.** Top-3 hit rate and ordering correlation beat plain accuracy here.
- F1 is noisy and grid-dominated — a strong baseline (e.g. rank by championship points) is hard to beat.
  Compare against one.

> ⚠️ Experimental / learning project. Not betting advice. Predictions stay off the personal-site widget.

### Setup
```bash
pip install -e '.[predictions]'                 # from repo root, in your venv
python -m ipykernel install --user --name f1pitwall   # then pick this kernel
```

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from f1pitwall.predictions import (
    build_predictions,
    export_predictions,
    latest_driver_lineup,
    load_results,
)

CACHE = Path("../python/.fastf1-cache")   # FastF1 telemetry/results cache (git-ignored)
PUBLIC_DATA = Path("../public-data")      # where the dashboard reads generated JSON

## 1. Load data

One row per (race, driver): `grid`, `finish`, `dnf`, `points`, plus a chronological `chrono` key.
The first run downloads and caches; later runs are fast.

In [ ]:
results = load_results([2024, 2025, 2026], CACHE)
print(results.shape)
results.head()

## 2. Feature engineering — *your work*

Build **pre-race** features only. The starter below shows the no-leakage pattern (`shift(1)` before the
rolling window). Extend it with the features you want: recent grid, DNF rate, constructor strength,
points-to-date, per-circuit history, weather, etc.

In [ ]:
df = results.sort_values("chrono").copy()
by_driver = df.groupby("driver_number", group_keys=False)

# Example: recent finishing form (mean of the last 3 finishes, current race excluded).
df["form_finish"] = by_driver["finish"].transform(
    lambda s: s.shift(1).rolling(3, min_periods=1).mean()
)

# TODO: add more features, e.g.
#   df["form_grid"]        = by_driver["grid"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
#   df["dnf_rate"]         = by_driver["dnf"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
#   df["points_to_date"]   = by_driver["points"].transform(lambda s: s.cumsum().shift(1)).fillna(0)
#   df["constructor_form"] = df.groupby("team", group_keys=False)["finish"].transform(
#                                lambda s: s.shift(1).rolling(6, min_periods=1).mean())

df["top3"] = (df["finish"] <= 3).astype(float)     # prediction target
FEATURES = ["form_finish"]                          # TODO: extend to your full feature set
df[["race_name", "code", "team", "grid", "finish", *FEATURES, "top3"]].head(12)

## 3. Train & evaluate — *your work*

Split **race-wise** and report **rank-aware** metrics. Compare against a baseline (e.g. rank by
`points_to_date`) — if the model can't beat it, say so.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupKFold

train = df[df["top3"].notna() & df[FEATURES].notna().all(axis=1)].copy()
X = train[FEATURES].fillna(train[FEATURES].median())
y = train["top3"]
groups = train["chrono"]   # one group per race → GroupKFold tests on unseen races

# TODO: cross-validate with GroupKFold and compute a top-3 hit rate per held-out race
#       (of the 3 highest-probability drivers, how many actually finished top 3?).
#       Also try ROC-AUC / log-loss, and a points-to-date baseline for comparison.

model = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
model.fit(X, y)
print("trained on", len(train), "driver-races across", train["chrono"].nunique(), "races")

## 4. Predict the next race — *your work*

Before a race weekend the entry list isn't published, so use the most recent lineup and each driver's
season-to-date form. Produce a **podium probability** per driver.

In [ ]:
lineup = latest_driver_lineup(results)

# TODO: build one feature row per driver from their latest season-to-date values,
#       then predict: lineup["podium_probability"] = model.predict_proba(rows)[:, 1]

# Placeholder probabilities so the export cell runs end-to-end — REPLACE with your model's output.
lineup = lineup.assign(podium_probability=np.linspace(0.85, 0.03, len(lineup)))
lineup.head()

## 5. Export to the dashboard

`build_predictions` validates against the shared contract; `export_predictions` writes
`public-data/predictions/next.json`. Run the dashboard (`pnpm --filter @f1pitwall/web dev`) and the
**Experimental Predictions** panel will render it.

In [ ]:
preds = build_predictions(
    year=2026,
    race_name="Belgian Grand Prix",
    round=10,
    model="your-model-name",          # name your approach
    drivers=lineup.to_dict("records"),
)
path = export_predictions(preds, PUBLIC_DATA)
print("wrote", path)